In [1]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import sklearn as skl
import gsw
import cartopy.crs as ccrs
from scipy.interpolate import interp1d
import os
import warnings
import joblib
warnings.filterwarnings('ignore')

In [2]:
os.system('echo $USER > userid')
usrid=np.genfromtxt('userid',dtype='<U16')
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/ML4O2_temp/')
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/ML4O2_results/')
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/WOD18_OSDCTD/')
#
# version information
# ver = np.genfromtxt('data_XXX.txt',dtype='U11').tolist()
ver = '2.2.2.4.2.4'
date1='07082025' # Set this for saving today's date. Usually date1=today's date
date2='07082025' # Set alternative date for re-running previous results
rerun = False    # indicate again whether you are re-running previous results
#
dirout=f'/glade/derecho/scratch/{usrid}/ML4O2_temp/'
#
print(f'-----------------------------------------------')
print(f' Machine Learning For Dissolved Oxygen (ML4O2) ')
print(f' version{ver} date:{date1}')
print(f'-----------------------------------------------')

-----------------------------------------------
 Machine Learning For Dissolved Oxygen (ML4O2) 
 version2.2.2.4.2.4 date:07082025
-----------------------------------------------


In [3]:
selection = ver.split('.')
basin = ['Atlantic','Pacific','Indian','Southern','Arctic','Mediterranean']
#
if selection[0] == '1':
    print('Random Forst algorithm will be used.')
    alg = 'RF'
elif selection[0] == '2':
    print('Neural Network algorithm will be used.')
    alg = 'NN'
else:
    print('error - incorrect algorithm type')
#
if selection[1] == '1':
    print('Ship-based O2 data will be used. ')
elif selection[1] == '2':
    print('Ship-based and Argo-O2 data will be used. ')
else:
    print('error - incorrect input data type')
#
if selection[2] == '1':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '2':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '3':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '4':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '5':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '6':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
else:
    print('error - incorrect O2 data type')
#
if selection[3] == '1':
    print('EN4 dataset will be used for T/S input. ')
    endyear=2021
    if selection[1]=='1':
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
elif selection[3] == '2':
    print('ORAS4 dataset will be used for T/S input. ')
    endyear=2018
elif selection[3] == '3':
    print('SODA3.4.2 dataset will be used for T/S input. ')
elif selection[3] == '4':
    print('EN4 (C14) dataset will be used for T/S input. ')
    endyear=2023
    if selection[1]=='1':
        #dirinput = f'/glade/work/ito/ML4O2/apr2025/input_202504_ship/'
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
    elif selection[1]=='2':
        #dirinput = f'/glade/work/ito/ML4O2/apr2025/input_202504_noadj/'
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
    elif selection[1]=='3':
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
    Nlev=63
else:
    print('error - incorrect T/S data type')
#
if selection[4] == '1':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, month')
elif selection[4] == '2':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)')
elif selection[4] == '3':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma')
elif selection[4] == '4':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma, N2')
#NEW
elif selection[4] == '5':
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month)')
elif selection[4] == '6':
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month)')
elif selection[4] == '7':
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, year, cos(month), sin(month)')
#NEW
else:
    print('error - incorrect predictor variable type')
#
if selection[5] == '1':
    print('Hyperparameter set is optimized via K-fold CV')
elif selection[5] == '2':
    print('A pre-set hyperparameter set is used')
elif selection[5] == '4':
    print('New K-fold cross validation')
else:
    print('error - incorrect hyperparameter type')

Neural Network algorithm will be used.
Ship-based and Argo-O2 data will be used. 
Pacific Ocean will be mapped
EN4 (C14) dataset will be used for T/S input. 
Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)
New K-fold cross validation


In [4]:
diro = f'/glade/derecho/scratch/{usrid}/WOD18_OSDCTD/'
if selection[3] == 1:
    dirf = '/glade/campaign/univ/ugit0034/EN4/L09_20x180x360/'
elif selection[3] == 2:
    dirf = '/glade/campaign/univ/ugit0034/ORAS4/TSN2/'
#
dirin = '/glade/campaign/univ/ugit0034/WOD18_OSDCTD/'
fargo = '/glade/campaign/univ/ugit0034/bgcargo/o2_Global_ARGO_Type12_47lev.nc'
fosd='_1x1bin_osd_'
fctd='_1x1bin_ctd_'
fmer='_1x1bin_osdctd_'
var=['o2','TSN2']
os.system('mkdir -p '+diro)
os.system('mkdir -p '+diro+'temp')

0

In [5]:
ds=xr.open_dataset(dirin+var[0]+fmer+str(1965)+'.nc')
Z=ds.depth.to_numpy()
Nz=np.size(Z)


# %%


# select analysis period
# do not change the start year from 1965 (this is when Carpenter 1965 established modern Winkler method)
yrs=np.arange(1965,endyear,1)

In [6]:
dsm=xr.open_dataset('/glade/campaign/univ/ugit0034/wod18/basin_mask_01.nc')
#
if selection[2] == '1':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='atlantic'
elif selection[2] == '2':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='pacific'
elif selection[2] == '3':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='indian'
elif selection[2] == '4':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='southern'
elif selection[2] == '5':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='arctic'
elif selection[2] == '6':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='mediterranean'
else:
    print('error - incorrect O2 data type')
#
if selection[3]=='1':
    if selection[1]=='2':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/o20_{bname0}_1x1_47lev.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/t0_{bname0}_1x1_47lev.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/s0_{bname0}_1x1_47lev.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/lon0_{bname0}_1x1_47lev.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/lat0_{bname0}_1x1_47lev.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/depth0_{bname0}_1x1_47lev.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/time0_{bname0}_1x1_47lev.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/month0_{bname0}_1x1_47lev.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/sigma0_{bname0}_1x1_47lev.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/N20_{bname0}_1x1_47lev.npy')
    elif selection[1]=='1':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/o20_{bname0}_1x1_47lev_ship.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/t0_{bname0}_1x1_47lev_ship.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/s0_{bname0}_1x1_47lev_ship.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/lon0_{bname0}_1x1_47lev_ship.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/lat0_{bname0}_1x1_47lev_ship.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/depth0_{bname0}_1x1_47lev_ship.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/time0_{bname0}_1x1_47lev_ship.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/month0_{bname0}_1x1_47lev_ship.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/sigma0_{bname0}_1x1_47lev_ship.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/N20_{bname0}_1x1_47lev_ship.npy')
    if (selection[4]=='6')&(selection[1]=='1'):
        doa1 = np.load(f'{dirinput}/o20_{bname0}_1x1_47lev.npy')
        dta1 = np.load(f'{dirinput}/t0_{bname0}_1x1_47lev.npy')
        dsa1 = np.load(f'{dirinput}/s0_{bname0}_1x1_47lev.npy')
        xx1 = np.load(f'{dirinput}/lon0_{bname0}_1x1_47lev.npy')
        yy1 = np.load(f'{dirinput}/lat0_{bname0}_1x1_47lev.npy')
        zz1 = np.load(f'{dirinput}/depth0_{bname0}_1x1_47lev.npy')
        tt1 = np.load(f'{dirinput}/time0_{bname0}_1x1_47lev.npy')
        tc1 = np.load(f'{dirinput}/month0_{bname0}_1x1_47lev.npy') 
        GMT1 = np.load(f'{dirinput}/GMT0_{bname0}_1x1_47lev.npy') 
#
elif selection[3]=='2':
    if selection[1]=='2':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/o20_{bname0}_1x1_47lev.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/t0_{bname0}_1x1_47lev.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/s0_{bname0}_1x1_47lev.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lon0_{bname0}_1x1_47lev.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lat0_{bname0}_1x1_47lev.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/depth0_{bname0}_1x1_47lev.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/time0_{bname0}_1x1_47lev.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/month0_{bname0}_1x1_47lev.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/sigma0_{bname0}_1x1_47lev.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/N20_{bname0}_1x1_47lev.npy')
    elif selection[1]=='1':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/o20_{bname0}_1x1_47lev_ship.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/t0_{bname0}_1x1_47lev_ship.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/s0_{bname0}_1x1_47lev_ship.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lon0_{bname0}_1x1_47lev_ship.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lat0_{bname0}_1x1_47lev_ship.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/depth0_{bname0}_1x1_47lev_ship.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/time0_{bname0}_1x1_47lev_ship.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/month0_{bname0}_1x1_47lev_ship.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/sigma0_{bname0}_1x1_47lev_ship.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/N20_{bname0}_1x1_47lev_ship.npy')
elif selection[3]=='4':
    doa1 = np.load(f'{dirinput}/o20_{bname0}_1x1_63lev.npy')
    dta1 = np.load(f'{dirinput}/t0_{bname0}_1x1_63lev.npy')
    dsa1 = np.load(f'{dirinput}/s0_{bname0}_1x1_63lev.npy')
    xx1 = np.load(f'{dirinput}/lon0_{bname0}_1x1_63lev.npy')
    yy1 = np.load(f'{dirinput}/lat0_{bname0}_1x1_63lev.npy')
    zz1 = np.load(f'{dirinput}/depth0_{bname0}_1x1_63lev.npy')
    tt1 = np.load(f'{dirinput}/time0_{bname0}_1x1_63lev.npy')
    tc1 = np.load(f'{dirinput}/month0_{bname0}_1x1_63lev.npy')
    GMT1 = np.load(f'{dirinput}/GMT0_{bname0}_1x1_63lev.npy')

# %%

Nsample = np.size(doa1)
print(Nsample)


Pacific Ocean will be mapped
4630957


In [7]:
if selection[4] == '1':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, tc1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, month')
elif selection[4] == '2':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)')
elif selection[4] == '3':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12), dsga1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma')
elif selection[4] == '4':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12), dsga1, dn2a1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma, N2')
elif selection[4] == '5':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month)')
elif selection[4] == '6':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, GMT1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month)')
elif selection[4] == '7':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, GMT1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, GMT, cos(month), sin(month)')
else:
    print('error - incorrect predictor variable type')

Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)


In [8]:
y = doa1
#
Xm = np.mean(X,axis=1)
Xstd = np.std(X,axis=1)
#
N=np.size(y)
# normalize x and y
Xa = (X.T - Xm)/Xstd
ym = np.mean(y)
ystd = np.std(y)
ya = (y-ym)/ystd
#
np.savez(dirout+f'ML_params_v{ver}.npz',Xm=Xm,Xstd=Xstd,ym=ym,ystd=ystd)

In [9]:
if rerun==False:
    yr_drop = np.random.choice(yrs,11,replace=False)
    print(yr_drop)

# group these years together into a single array
if rerun==False:
    yr1=np.round(tt1/12+1965)
    ind=(yr1==int(yr_drop[0]))
    N=np.sum(ind)
    for n in np.arange(1,11,1):
        tmp=(yr1==yr_drop[n])
        ind=(ind==True)|(tmp==True)
        N=N+np.sum(tmp)
    #print(N,np.sum(ind))
    print(f'the count of data point (bins) = {yr1.size}')
    print(f'the count of test data (bins) = {N}, which is {N/yr1.size*100}%')

# Assemble into input data (train/test) and save it for record
if rerun==False:
    ind1 = (ind==False)
    X_train = Xa[ind1,:]
    X_test = Xa[ind,:]
    y_train = ya[ind1]
    y_test = ya[ind]
    print(f'the count of train data point (bins) = {y_train.size}')
    np.savez(dirout+f'train_test_v{ver}_{date1}.npz',X_train=X_train,X_test=X_test,
             y_train=y_train,y_test=y_test,yr_drop=yr_drop)

[2015 1989 2002 2010 2020 1988 1980 1986 1997 1999 1970]
the count of data point (bins) = 4630957
the count of test data (bins) = 872049, which is 18.830859366649268%
the count of train data point (bins) = 3758908


In [10]:
tmp=np.load(dirout+f'train_test_v{ver}_{date2}.npz')
X_train = tmp['X_train']
X_test  = tmp['X_test']
y_train = tmp['y_train']
y_test  = tmp['y_test']
#
tmp = np.load(dirout+f'ML_params_v{ver}.npz')
Xstd = tmp['Xstd']
Xm   = tmp['Xm']

In [11]:
if (selection[4] == '5')|(selection[4] == '6'):
    ttmp0 = tt1[ind1]
else:
    ttmp0 = X_train[:,5]*Xstd[5]+Xm[5]
#
yr1 = ttmp0/12+1965

# Calculate the Decadal Group K-fold
tbnds=[1965,1977,1989,2000,2012,2022]
Kval = 5
# yr1=X[5,ind1]/12+1965
print(f'The total count of data points = {yr1.size}')

The total count of data points = 3758908


In [12]:
for n in range(5):
    K_test=(yr1>=tbnds[n])&(yr1<tbnds[n+1])
    K_train=(K_test==False)
    X_trainK = X_train[K_train,:]
    X_testK = X_train[K_test,:]
    y_trainK = y_train[K_train]
    y_testK = y_train[K_test]
    # check
    print(f'N,train = {y_train.size}, Group {n} train size = {y_trainK.size}, Group {n} test size = {y_testK.size}, {y_testK.size/y_train.size*100}%')


# ### Algorithm selection & training

# %%


RF_parameters = {'min_samples_split':[2,4,8,16,32,64],'max_features':[2,3,5]}
NN_parameters = {'hidden_layer_sizes':[[5,5],[10,10],[20,20],[40,40],
                                       [20,10],[20,15,10,5],[20,15,10,5,5,5]],'alpha':[.001, .01, .1]}

N,train = 3758908, Group 0 train size = 2887582, Group 0 test size = 871326, 23.180295979577046%
N,train = 3758908, Group 1 train size = 3041435, Group 1 test size = 717473, 19.087272154572553%
N,train = 3758908, Group 2 train size = 3304983, Group 2 test size = 453925, 12.075980577337885%
N,train = 3758908, Group 3 train size = 3092834, Group 3 test size = 666074, 17.719880348228795%
N,train = 3758908, Group 4 train size = 2869767, Group 4 test size = 889141, 23.6542368155858%


In [13]:
def train_K(k):
    if alg =='RF':
        from sklearn.ensemble import RandomForestRegressor
        mxf=RF_parameters['max_features'][parm2]
        msp=RF_parameters['min_samples_split'][parm1]
        msl=5
        nest=500
        regr=RandomForestRegressor(n_jobs=-1,n_estimators=nest,min_samples_split=msp,
                                   min_samples_leaf=msl,max_features=mxf)
        K_test=(yr1>=tbnds[k])&(yr1<tbnds[k+1])
        K_train=(K_test==False)
        X_trainK = X_train[K_train,:]
        X_testK = X_train[K_test,:]
        y_trainK = y_train[K_train]
        y_testK = y_train[K_test]
        regr.fit(X_trainK, y_trainK)
        y_est = regr.predict(X_testK)
        np.savez(dirout+f'RFtest_pred_v{ver}_cv{k}_{parm1}_{parm2}.npz',Xtest=X_testK,test=y_testK,est=y_est)
        filename = dirout+f'algorithm_v{ver}_cv{k}_{parm1}_{parm2}.sav'
        joblib.dump(regr, filename)
    elif alg == 'NN':
        from sklearn.neural_network import MLPRegressor
        hls=NN_parameters['hidden_layer_sizes'][parm1]
        alp=NN_parameters['alpha'][parm2]
        regr=MLPRegressor(max_iter=1000,hidden_layer_sizes=hls,alpha=alp)
        K_test=(yr1>=tbnds[k])&(yr1<tbnds[k+1])
        K_train=(K_test==False)
        X_trainK = X_train[K_train,:]
        X_testK = X_train[K_test,:]
        y_trainK = y_train[K_train]
        y_testK = y_train[K_test]
        regr.fit(X_trainK, y_trainK)
        y_est = regr.predict(X_testK)
        np.savez(dirout+f'NNtest_pred_v{ver}_cv{k}_{parm1}_{parm2}.npz',Xtest=X_testK,test=y_testK,est=y_est)
        filename = dirout+f'algorithm_v{ver}_cv{k}_{parm1}_{parm2}.sav'
        joblib.dump(regr, filename)
    r=np.corrcoef(y_est,y_testK)
    return np.round(r[0,1]**2,4)

In [14]:
for parm1 in range(6):
#for parm1 in [5]:
    for parm2 in range(3):
    #for parm2 in [1,2]:
        if alg =='NN':
            from multiprocessing import Pool
            if __name__ == '__main__':
                with Pool(5) as p:
                    print(p.map(train_K, [0, 1, 2, 3, 4]))
        elif alg=='RF':
            for n in range(5):
                r2=train_K(n)
                print(n,parm1,parm2,r2)

[np.float64(0.9016), np.float64(0.9155), np.float64(0.904), np.float64(0.935), np.float64(0.9438)]
[np.float64(0.9053), np.float64(0.9173), np.float64(0.9043), np.float64(0.9349), np.float64(0.9369)]
[np.float64(0.9063), np.float64(0.9137), np.float64(0.9088), np.float64(0.9331), np.float64(0.935)]
[np.float64(0.9262), np.float64(0.9395), np.float64(0.9352), np.float64(0.9473), np.float64(0.9543)]
[np.float64(0.9285), np.float64(0.9397), np.float64(0.9327), np.float64(0.9488), np.float64(0.954)]
[np.float64(0.9251), np.float64(0.9352), np.float64(0.9272), np.float64(0.9452), np.float64(0.952)]
[np.float64(0.9418), np.float64(0.9468), np.float64(0.9416), np.float64(0.9564), np.float64(0.9613)]
[np.float64(0.9385), np.float64(0.9466), np.float64(0.9417), np.float64(0.9553), np.float64(0.9634)]
[np.float64(0.9308), np.float64(0.9411), np.float64(0.9337), np.float64(0.9499), np.float64(0.956)]
[np.float64(0.9446), np.float64(0.9513), np.float64(0.9491), np.float64(0.96), np.float64(0.9667)